# HW2: benchmark batch sizes and continue our model
Run sections separately. Start on CPU. This notebook requires the verified prior run already saved in MyDrive/HSE_NLP_HW2. It pins the actual training source. Existing run names deliberately stop duplicate launches; choose a new run name for an intentional repeat.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU before preparing the continuation'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
print('CPU_DRIVE_READY', Path('/content/drive').is_mount())

In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, tarfile, subprocess, io
assert not Path('/dev/nvidia0').exists()
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
repo=Path('/content/hw2-continuation-source')
assert not repo.exists()
subprocess.run(['git','init',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--depth=1','https://github.com/OsVox/hse-nlp.git','a411a64f8afe61e3a2b337af338c0ea6c19c9c15'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','FETCH_HEAD'],check=True)
os.chdir(repo)
commit=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
assert commit.startswith('a411a64')
subprocess.run(['pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import sha256, drive_preflight
from homeworks.hw2.prepare_colab import _copy_verified
drive_preflight(root)
record=json.loads((root/'prepared/ready.json').read_text())
cache=Path('/content/hw2_cache');cache.mkdir(exist_ok=True)
archive=Path('/content/data.tar')
shutil.copyfile(root/'prepared/data.tar',archive)
assert sha256(archive)==record['data_sha256']
with tarfile.open(archive) as z:z.extractall(cache,filter='data')
assert all(sha256(cache/name)==digest for name,digest in record['parquet_files'].items())
parent=root/'runs/mixed-lr001-20k-drive-v1/step-020000'
manifest=json.loads((parent/'checkpoint.complete.json').read_text())
local_parent=Path('/content/parent-020000');local_parent.mkdir(exist_ok=True)
for name,digest in manifest['files'].items():
    shutil.copyfile(parent/name,local_parent/name)
    assert sha256(local_parent/name)==digest
shutil.copyfile(parent/'checkpoint.complete.json',local_parent/'checkpoint.complete.json')
assert sha256(local_parent/'model.safetensors')=='e6f632ec0edaa2969493d1d6f8543de2cb558316881d13cd34f5ef4fa6bda0d7'
prepared=root/'prepared/continuation-v1';prepared.mkdir(exist_ok=True)
from homeworks.hw2.benchmark_batches import prepare
pool=Path('/content/probe.jsonl');prepare(cache,pool)
_copy_verified(pool,prepared/'probe.jsonl')
bundle=Path('/content/continuation-source.tar')
with tarfile.open(bundle,'w') as z:
    payload=(commit+'\n').encode(); info=tarfile.TarInfo('source_commit.txt');info.size=len(payload);z.addfile(info,io.BytesIO(payload))
    for p in sorted(Path('homeworks/hw2').glob('*.py')):z.add(p,arcname=str(p))
    for p in sorted(Path('homeworks/hw2/tokenizer').glob('*')):
        if p.is_file():z.add(p,arcname=str(p))
_copy_verified(bundle,prepared/'source.tar')
ready={'source_commit':commit,'source_sha256':sha256(prepared/'source.tar'),'pool_sha256':sha256(prepared/'probe.jsonl'),'parent':str(parent),'parent_model_sha256':sha256(local_parent/'model.safetensors'),'data_sha256':record['data_sha256'],'test_rows':1919,'gpu_allocated':False}
(prepared/'ready.json').write_text(json.dumps(ready,indent=2))
print('CPU_CONTINUATION_READY',ready)


## Switch to G4 only after CPU_CONTINUATION_READY
The GPU restore reads verified inputs from Drive, then starts a bounded two-pass benchmark. It compares identical examples at batch sizes 64, 128, 256 and 512. This is a throughput probe, not a quality comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive',timeout_ms=300000)
from pathlib import Path
import os,json,hashlib,shutil,tarfile,subprocess
root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
prepared=root/'prepared/continuation-v1'
ready=json.loads((prepared/'ready.json').read_text())
assert ready['gpu_allocated'] is False
repo=Path('/content/hw2-continuation-source');repo.mkdir(exist_ok=True)
bundle=Path('/content/continuation-source.tar')
shutil.copyfile(prepared/'source.tar',bundle)
assert hashlib.sha256(bundle.read_bytes()).hexdigest()==ready['source_sha256']
with tarfile.open(bundle) as z:z.extractall(repo,filter='data')
os.chdir(repo)
subprocess.run(['pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import sha256,drive_preflight
drive_preflight(root)
cache=Path('/content/hw2_cache');cache.mkdir(exist_ok=True)
record=json.loads((root/'prepared/ready.json').read_text())
if not all((cache/name).exists() and sha256(cache/name)==digest for name,digest in record['parquet_files'].items()):
    archive=Path('/content/data.tar');shutil.copyfile(root/'prepared/data.tar',archive)
    assert sha256(archive)==ready['data_sha256']
    with tarfile.open(archive) as z:z.extractall(cache,filter='data')
assert all(sha256(cache/name)==digest for name,digest in record['parquet_files'].items())
parent=Path(ready['parent']);manifest=json.loads((parent/'checkpoint.complete.json').read_text())
local_parent=Path('/content/parent-020000');local_parent.mkdir(exist_ok=True)
for name,digest in manifest['files'].items():
    if not (local_parent/name).exists() or sha256(local_parent/name)!=digest:shutil.copyfile(parent/name,local_parent/name)
    assert sha256(local_parent/name)==digest
shutil.copyfile(parent/'checkpoint.complete.json',local_parent/'checkpoint.complete.json')
pool=Path('/content/probe.jsonl');shutil.copyfile(prepared/'probe.jsonl',pool)
assert sha256(pool)==ready['pool_sha256']
import torch
print('GPU_RESTORE_READY',torch.cuda.get_device_name(0),ready['source_commit'])
probe_root=root/'benchmarks/batches-from-20k-v1';probe_root.mkdir(parents=True,exist_ok=True)
assert not (probe_root/'probe.started.json').exists(),'Inspect existing benchmark before repeating'
with (probe_root/'probe.log').open('w') as output:
    probe_process=subprocess.Popen(['python','-u','-m','homeworks.hw2.benchmark_batches','--checkpoint',str(local_parent),'--pool',str(pool),'--output',str(probe_root)],stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
print('PROBE_PID',probe_process.pid)


In [ ]:
import json,subprocess
print('PROBE_RETURN_CODE',probe_process.poll())
print((probe_root/'probe.log').read_text()[-4500:])
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw','--format=csv'],text=True))


## Start the longer run after the probe exits successfully
Select the fastest measured batch that leaves memory headroom. Resume our own 20,000-update model and optimizer, warm up a new learning-rate phase, and train to update 60,000. Every 5,000 updates save weights, optimizer state and all 1,919 exact predictions to Drive. Export and score every saved JSONL with the course bot. Monitor runtime release; the automatic release request can fail and then requires manual disconnect.

In [ ]:
import json, statistics, subprocess, datetime, threading, time
from pathlib import Path
from zipfile import ZipFile,ZIP_DEFLATED
assert probe_process.poll()==0,'Wait for the bounded benchmark'
benchmark=json.loads((probe_root/'benchmark.json').read_text())
rows=benchmark['results']
summary=[]
for batch_size in [64,128,256,512]:
    measured=[r for r in rows if r['batch_size']==batch_size and r['status']=='ok']
    if len(measured)!=2:continue
    samples=[]
    for r in measured:
        for line in (probe_root/f"gpu-round{r['round']}-batch{batch_size}.csv").read_text().splitlines():
            parts=line.split(',')
            if len(parts)==5:
                try:samples.append(float(parts[1]))
                except ValueError:pass
    summary.append({'batch_size':batch_size,'examples_per_second':statistics.median(r['examples_per_second'] for r in measured),'peak_gib':max(r['peak_allocated_gib'] for r in measured),'gpu_util_mean':statistics.mean(samples) if samples else None,'preparation_fraction':statistics.mean(r['preparation_seconds']/r['seconds'] for r in measured)})
total_gib=torch.cuda.get_device_properties(0).total_memory/2**30
eligible=[r for r in summary if r['peak_gib']<.7*total_gib]
assert eligible
choice=max(eligible,key=lambda r:r['examples_per_second'])
print('BATCH_MEASUREMENTS',json.dumps(summary))
print('CHOSEN_BATCH',choice)
selection={'summary':summary,'chosen':choice,'rule':'highest median examples/second from two rounds, peak memory below 70% of GPU capacity'}
(probe_root/'selection.json').write_text(json.dumps(selection,indent=2))
RUN=f"continue-20k-60k-b{choice['batch_size']}-v1"
run_root=root/'runs'/RUN
assert not run_root.exists(),'Inspect existing continuation before launching again'
run_root.mkdir(parents=True)
command=['python','-u','-m','homeworks.hw2.train','--resume-from',str(local_parent),'--restart-schedule','--steps','60000','--batch-size',str(choice['batch_size']),'--accumulation','1','--precision','bf16','--group-window','32','--mixed-rows-per-shard','4096','--learning-rate','0.0005','--warmup','500','--save-every','5000','--eval-every','5000','--cache-dir',str(cache),'--output-dir',f'/content/{RUN}','--drive-root',str(run_root),'--predict-at-checkpoint']
(run_root/'launch.json').write_text(json.dumps({'command':command,'source_commit':ready['source_commit'],'parent_model_sha256':ready['parent_model_sha256'],'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'gpu':torch.cuda.get_device_name(0),'batch_selection':selection},indent=2))
with (run_root/'training.log').open('w') as output:
    training_process=subprocess.Popen(command,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
telemetry_output=(run_root/'gpu-utilization.csv').open('w')
gpu_monitor=subprocess.Popen(['nvidia-smi','--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,power.draw','--format=csv,noheader,nounits','--loop=1'],stdout=telemetry_output,stderr=subprocess.DEVNULL)
def finish_continuation():
    rc=training_process.wait()
    gpu_monitor.terminate();gpu_monitor.wait();telemetry_output.close()
    markers=sorted(run_root.glob('step-*/results.complete.json'))
    with ZipFile(run_root/'predictions-and-evidence.zip','w',ZIP_DEFLATED) as z:
        for marker in markers:
            folder=marker.parent
            for name in ['predictions.csv','predictions.jsonl','predictions.provenance.json','results.complete.json','validation.json','checkpoint.complete.json','run_config.json']:
                if (folder/name).exists():z.write(folder/name,arcname=f'{folder.name}/{name}')
        for name in ['launch.json','training.log','gpu-utilization.csv']:z.write(run_root/name,arcname=name)
    (run_root/'execution.complete.json').write_text(json.dumps({'returncode':rc,'checkpoints':len(markers),'finished_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()},indent=2))
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        (run_root/'release-error.txt').write_text(str(e))
watcher=threading.Thread(target=finish_continuation,daemon=True);watcher.start()
print('LONGER_TRAINING_STARTED',RUN,training_process.pid)


In [ ]:
print('TRAIN_RETURN_CODE',training_process.poll())
print((run_root/'training.log').read_text()[-1800:])
print('READY',[p.parent.name for p in sorted(run_root.glob('step-*/results.complete.json'))])
print(subprocess.check_output(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total,power.draw','--format=csv'],text=True))
